## 1) 의존성 설치

In [1]:
!pip -q install -U "transformers>=4.44,<4.46" "accelerate>=0.33" "peft>=0.12" "tqdm" \
  "sacrebleu>=2.4" "rouge-score>=0.1.2"
!pip -q install sacrebleu rouge-score konlpy python-mecab-ko

## 2) 유틸 + 데이터 로더 (GitHub URL → raw 변환 포함)

In [2]:
import json, re, pathlib, requests
from typing import List, Dict
from tqdm import tqdm

def to_raw_github_url(url: str) -> str:
    """
    https://github.com/.../blob/main/path/file.json  →
    https://raw.githubusercontent.com/.../main/path/file.json
    """
    url = url.strip()
    url = url.replace("https://github.com/", "https://raw.githubusercontent.com/")
    url = url.replace("/blob/", "/")
    return url

def load_json_from_github(url: str) -> List[Dict]:
    raw_url = to_raw_github_url(url)
    r = requests.get(raw_url)
    r.raise_for_status()
    return r.json()

DEV_URL  = "https://github.com/beefed-up-geek/HCLT-KACL-2025/blob/main/Korean_Dialogue_Summarization/dataset/original_english_formatted/dev.json"
TEST_URL = "https://github.com/beefed-up-geek/HCLT-KACL-2025/blob/main/Korean_Dialogue_Summarization/dataset/original_formatted/test.json"

dev_data  = load_json_from_github(DEV_URL)
test_data = load_json_from_github(TEST_URL)

print("Loaded dev:", len(dev_data))
print("Loaded test:", len(test_data))
print("Dev example keys:", list(dev_data[0].keys()))


Loaded dev: 102
Loaded test: 408
Dev example keys: ['id', 'dialogue', 'subject_keyword', 'speaker_map', 'output']


## 3) 모델/토크나이저 로드 (GPU 자동 할당)

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

## 4) 프롬프트 템플릿 + 생성 함수

In [4]:
# === 0) 유틸 ===
from typing import Dict, Any, List
import torch

def normalize_dialogue(dlg: str) -> str:
    """
    간단 정규화: 양끝 공백/빈줄 제거, Windows 개행 -> \n
    필요 시 추가 정제 규칙을 여기에 확장.
    """
    if not isinstance(dlg, str):
        return ""
    t = dlg.replace("\r\n", "\n").replace("\r", "\n").strip()
    # 연속 빈 줄 2개 이상 -> 하나
    while "\n\n\n" in t:
        t = t.replace("\n\n\n", "\n\n")
    return t


# === 1) system / user 메시지 생성 (요청 형식 그대로) ===
'''def build_system_and_user(sample: Dict[str, Any]) -> Dict[str, str]:
    """
    system / user 메시지를 분리해 반환합니다.
    - system: 요약 지침
    - user  : 실제 대화 텍스트
    """
    dlg = normalize_dialogue(sample.get("dialogue"))
    subjects = sample.get("subject_keyword") or []
    subj_txt = ", ".join(map(str, subjects)) if subjects else ""

    instr = (
        "당신은 두 사람의 대화를 바탕으로 문제를 해결하는 AI 어시스턴트입니다. \n"
        "[규칙]\n "
        "- 대화를 주의 깊게 읽고 문맥을 이해한 뒤, 딱 한 문단으로 간결하고 사실적으로 요약하세요. \n"
        "- 먼저 전체적인 주제를 한 문장으로 간략히 소개한 후, 이어서 화자 1이 말한 내용을 모두 모아 4-5문장으로 요약하고, 그 후에 화자 2가 말한 내용을 모두 모아 4-5문장으로 요약하세요. \n"
        "- 출력 예시는 다음과 같습니다. \n [예시] 두 화자는 이 대화에서 ~에 대해 말했습니다. 화자 1은 ~. 화자 2는 ~ \n"
    )

    if subj_txt:
        instr += (
            "\n- Reference topic(s): " + subj_txt +
            "\n- 위 주제와 직접 관련된 정보에 가중치를 두고, 주변 맥락은 간략히 처리하세요."
        )

    return {
        "system": instr.strip(),
        "user": dlg.strip()
    }'''

def build_system_and_user(sample: Dict[str, Any]) -> Dict[str, str]:
    """
    Returns separated system / user messages.
    - system: summarization instruction
    - user  : dialogue text
    """
    dlg = normalize_dialogue(sample.get("dialogue"))
    subjects = sample.get("subject_keyword") or []
    subj_txt = ", ".join(map(str, subjects)) if subjects else ""

    instr = (
        "You are an AI assistant. Read the dialogue and write a clear, factual summary in English.\n"
        "- Summarize in **one paragraph**.\n"
        "- Start with one sentence introducing the main topic.\n"
        "- Then summarize Speaker 1’s points in 4–5 sentences, followed by Speaker 2’s points in 4–5 sentences.\n"
        "- Example: In this dialogue, the two speakers discussed ~. Speaker 1 said ~. Speaker 2 said ~."
    )

    if subj_txt:
        instr += (
            f"\n- Reference topic(s): {subj_txt}. Give more weight to these topics and keep other context brief."
        )

    return {
        "system": instr.strip(),
        "user": dlg.strip()
    }

# === 2) chat 메시지 구성 ===
def build_messages(sample: Dict[str, Any]) -> List[Dict[str, str]]:
    su = build_system_and_user(sample)
    messages = [
        {"role": "system", "content": su["system"]},
        {"role": "user",   "content": su["user"]},
    ]
    return messages


# === 3) 생성 함수 (apply_chat_template 사용) ===
@torch.inference_mode()
def generate_summary_from_sample(
    sample: Dict[str, Any],
    max_new_tokens: int = 1024,
    temperature: float = 0.2,
    top_p: float = 0.9
) -> str:
    """
    sample(dict) 안의 'dialogue'와 'subject_keyword'를 사용해 요약 생성.
    tokenizer.apply_chat_template로 role 기반 포맷을 모델 입력으로 변환.
    """
    messages = build_messages(sample)

    # Llama3 계열은 chat 템플릿 제공 → 문자열 프롬프트 생성
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True  # assistant 답변 자리까지 포함
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # apply_chat_template를 썼기 때문에, 마지막 assistant 영역만 깔끔히 추출
    # 가장 간단한 방법: 프롬프트 문자열 길이 이후 부분을 잘라서 사용
    # (주의: 일부 모델은 토큰화 과정에서 공백/개행이 약간 달라질 수 있어 후처리를 추가)
    # 안전하게 마지막 occurrence 기준으로 잘라내기:
    cut = full_text.rfind(messages[-1]["content"])
    if cut != -1:
        # 유저 콘텐츠 직후부터가 아닐 수 있어서, 전체에서 프롬프트 길이 기반으로 재계산
        pass

    # 프롬프트 길이 기반 자르기 (일반적으로 가장 확실)
    gen_only = full_text[len(prompt_text):].strip()
    return gen_only if gen_only else full_text.strip()


# === 4) (선택) 기존 인터페이스 호환 wrapper ===
def generate_summary(dialogue: str, subject_keywords: List[str],
                     max_new_tokens: int = 512, temperature: float = 0.2, top_p: float = 0.9) -> str:
    """
    이전 코드와 호환되도록, (dialogue, subject_keywords) 시그니처를 유지한 wrapper.
    내부적으로 role 기반 메시지 경유.
    """
    sample = {"dialogue": dialogue, "subject_keyword": subject_keywords}
    return generate_summary_from_sample(
        sample,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p
    )


## 5) Dev 세트 평가 (ROUGE/ BLEU)
* ROUGE-1/2/Lsum
* BLEU (sacrebleu, 한국어에는 tokenize='char' 권장

In [5]:
from typing import List, Dict
from rouge_score import rouge_scorer
import sacrebleu
from sacrebleu.metrics import BLEU
import numpy as np
import pandas as pd

# ---------------------------
# ROUGE (변경 없음)
# ---------------------------
def calc_rouge(preds: List[str], refs: List[str]) -> Dict[str, float]:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeLsum"], use_stemmer=False)
    r1, r2, rl = [], [], []
    for p, r in zip(preds, refs):
        scores = scorer.score(r, p)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeLsum"].fmeasure)
    return {
        "rouge1": float(np.mean(r1)),
        "rouge2": float(np.mean(r2)),
        "rougeLsum": float(np.mean(rl)),
    }

# ---------------------------
# 영어용 BLEU-n (tokenize="13a")
# ---------------------------
def bleu_ngram_en(preds: List[str], refs: List[str], n: int = 4, lowercase: bool = False) -> float:
    """
    n: 1~4 (BLEU-1..4)
    lowercase=True 로 두면 대소문자 무시(뉴스/MT 세팅에 따라 옵션)
    """
    metric = BLEU(tokenize="13a", max_ngram=n, effective_order=True, lowercase=lowercase)
    return float(metric.corpus_score(preds, [refs]).score)

def calc_bleu_en_all(preds: List[str], refs: List[str], lowercase: bool = False) -> Dict[str, float]:
    """
    영어용 BLEU-1/2/3/4 점수를 각각 반환
    tokenize="13a" : 영어 평가 표준 토크나이저
    lowercase=True 로 두면 대소문자 무시
    """
    # BLEU metric 초기화 (항상 4-gram까지 계산)
    metric = BLEU(tokenize="13a", effective_order=True, lowercase=lowercase)
    result = metric.corpus_score(preds, [refs])

    # result.precisions = [p1, p2, p3, p4]
    return {
        "bleu1_en": float(result.precisions[0]),
        "bleu2_en": float(result.precisions[1]),
        "bleu3_en": float(result.precisions[2]),
        "bleu4_en": float(result.precisions[3]),  # 이것이 통상 BLEU라고 불리는 값
    }

# ---------------------------
# Dev inference (그대로)
# ---------------------------
dev_preds, dev_refs, dev_ids = [], [], []
for item in tqdm(dev_data, desc="Dev summarizing", unit="sample"):
    dialogue = item.get("dialogue", "")
    subject_kw = item.get("subject_keyword", []) or []
    ref = item.get("output", "")

    pred = generate_summary(dialogue, subject_kw, max_new_tokens=1024, temperature=0.2, top_p=0.9)

    dev_ids.append(item.get("id"))
    dev_refs.append(ref)
    dev_preds.append(pred)

# ---------------------------
# Metrics 출력
# ---------------------------
rouge = calc_rouge(dev_preds, dev_refs)
bleu_en = calc_bleu_en_all(dev_preds, dev_refs, lowercase=False)  # 필요시 True

print("=== Dev Metrics (EN) ===")
print(f"ROUGE-1   (F): {rouge['rouge1']:.4f}")
print(f"ROUGE-2   (F): {rouge['rouge2']:.4f}")
print(f"ROUGE-Ls  (F): {rouge['rougeLsum']:.4f}")

print(f"BLEU-1 (13a): {bleu_en['bleu1_en']:.2f}")
print(f"BLEU-2 (13a): {bleu_en['bleu2_en']:.2f}")
print(f"BLEU-3 (13a): {bleu_en['bleu3_en']:.2f}")
print(f"BLEU-4 (13a): {bleu_en['bleu4_en']:.2f}")


Dev summarizing: 100%|██████████| 102/102 [11:06<00:00,  6.53s/sample]


=== Dev Metrics (EN) ===
ROUGE-1   (F): 0.3191
ROUGE-2   (F): 0.0608
ROUGE-Ls  (F): 0.1701
BLEU-1 (13a): 33.85
BLEU-2 (13a): 6.54
BLEU-3 (13a): 1.96
BLEU-4 (13a): 0.71


## 6) Test 세트 제출물 생성 및 저장

In [6]:
'''# Test inference
test_outputs = []
for item in tqdm(test_data, desc="Test summarizing", unit="sample"):
    dialogue = item.get("dialogue", "")
    subject_kw = item.get("subject_keyword", []) or []
    pred = generate_summary(dialogue, subject_kw, max_new_tokens=256, temperature=0.2, top_p=0.9)
    test_outputs.append({
        "id": item.get("id"),
        "output": pred
    })

save_path = "submission_test_pred.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(test_outputs, f, ensure_ascii=False, indent=2)

print(f"Saved submission file to: {save_path}")
print(test_outputs[:2])  # 앞 2개 미리보기'''


'# Test inference\ntest_outputs = []\nfor item in tqdm(test_data, desc="Test summarizing", unit="sample"):\n    dialogue = item.get("dialogue", "")\n    subject_kw = item.get("subject_keyword", []) or []\n    pred = generate_summary(dialogue, subject_kw, max_new_tokens=256, temperature=0.2, top_p=0.9)\n    test_outputs.append({\n        "id": item.get("id"),\n        "output": pred\n    })\n\nsave_path = "submission_test_pred.json"\nwith open(save_path, "w", encoding="utf-8") as f:\n    json.dump(test_outputs, f, ensure_ascii=False, indent=2)\n\nprint(f"Saved submission file to: {save_path}")\nprint(test_outputs[:2])  # 앞 2개 미리보기'